In [ ]:
import requests
import os
from dotenv import load_dotenv
from modules.get_token import get_token
load_dotenv()
start_url = os.getenv("PRJ_START_URL")

In [ ]:
query = """
    query {
      __type(name: "Equipment") {
        fields(includeDeprecated: false) {
          name
          type {
            name
            kind
          }
        }
      }
    }
    """

headers = {
    "content-type": "application/json"
}
token = get_token()
url = f"{start_url}/graphql?token={token}"
res = requests.post(url, headers=headers, json={"query": query})
print(res.json()['data']["__type"].get("fields"))

[{'name': 'brand', 'type': {'kind': 'SCALAR', 'name': 'String'}}, {'name': 'bulkImportId', 'type': {'kind': 'SCALAR', 'name': 'String'}}, {'name': 'copiedFrom', 'type': {'kind': 'SCALAR', 'name': 'String'}}, {'name': 'copiedTime', 'type': {'kind': 'SCALAR', 'name': 'String'}}, {'name': 'createdBy', 'type': {'kind': 'OBJECT', 'name': 'User'}}, {'name': 'createdByPlainText', 'type': {'kind': 'SCALAR', 'name': 'String'}}, {'name': 'createdTime', 'type': {'kind': 'SCALAR', 'name': 'String'}}, {'name': 'description', 'type': {'kind': 'SCALAR', 'name': 'String'}}, {'name': 'editLog', 'type': {'kind': 'OBJECT', 'name': 'EditLog'}}, {'name': 'editLogPlainText', 'type': {'kind': 'SCALAR', 'name': 'String'}}, {'name': 'equipmentNumber', 'type': {'kind': 'SCALAR', 'name': 'String'}}, {'name': 'equipmentType', 'type': {'kind': 'SCALAR', 'name': 'String'}}, {'name': 'files', 'type': {'kind': 'NON_NULL', 'name': None}}, {'name': 'images', 'type': {'kind': 'NON_NULL', 'name': None}}, {'name': 'instal

In [ ]:

import json
import pandas as pd

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')


def introspect_equipment_schema(token):
    """Use GraphQL introspection to discover all available fields in Equipment/Tools"""
    url = f"{start_url}/graphql?token={token}"
    headers = {"content-type": "application/json"}
    
    # Try multiple possible type names based on the UI
    type_names = [
        "Equipment",
        "Tool", 
        "ToolRecord",
        "EquipmentRecord",
        "InternalTool",
        "Asset",
        "MachineTools"
    ]
    
    for type_name in type_names:
        logging.info(f"\nTrying type name: {type_name}...")
        
        introspection_query = f"""
        {{
            __type(name: "{type_name}") {{
                name
                kind
                fields {{
                    name
                    type {{
                        name
                        kind
                        ofType {{
                            name
                            kind
                        }}
                    }}
                }}
            }}
        }}
        """
        
        payload = {"query": introspection_query}
        
        try:
            res = requests.post(url, headers=headers, json=payload)
            res.raise_for_status()
            
            response_data = res.json()
            
            if "errors" in response_data:
                logging.debug(f"GraphQL errors: {response_data['errors']}")
                continue
            
            if "data" in response_data and response_data["data"]["__type"]:
                equipment_type = response_data["data"]["__type"]
                fields = equipment_type.get("fields", [])
                
                if fields:
                    logging.info(f"\n{'='*100}")
                    logging.info(f"✅ FOUND EQUIPMENT SCHEMA: {type_name}")
                    logging.info(f"{'='*100}\n")
                    
                    field_names = []
                    logging.info(f"{'#':<5} {'Field Name':<40} {'Type':<30}")
                    logging.info("-" * 100)
                    
                    for idx, field in enumerate(fields, 1):
                        field_name = field["name"]
                        field_type = field.get("type", {})
                        
                        # Extract type information
                        type_kind = field_type.get("kind", "")
                        type_name_str = field_type.get("name", "")
                        
                        if not type_name_str and field_type.get("ofType"):
                            type_name_str = field_type["ofType"].get("name", "Unknown")
                        
                        field_names.append(field_name)
                        logging.info(f"{idx:<5} {field_name:<40} {type_name_str:<30}")
                    
                    logging.info("=" * 100)
                    logging.info(f"Total fields available: {len(field_names)}\n")
                    
                    # Save field list
                    with open('equipment_discovered_fields.txt', 'w') as f:
                        f.write(f"Equipment Fields ({type_name})\n")
                        f.write("=" * 80 + "\n\n")
                        for idx, field_name in enumerate(field_names, 1):
                            f.write(f"{idx}. {field_name}\n")
                    
                    logging.info("📁 Field list saved to: equipment_discovered_fields.txt")
                    
                    return field_names, type_name
                    
        except Exception as e:
            logging.debug(f"Error trying {type_name}: {e}")
            continue
    
    logging.error("❌ Could not find Equipment schema with any type name")
    
    # Try to discover what queries are available
    logging.info("\nSearching for equipment-related queries...")
    discover_equipment_queries(token, url, headers)
    
    return None, None


def discover_equipment_queries(token, url, headers):
    """Discover all available query endpoints to find equipment-related ones"""
    introspection_query = """
    {
        __schema {
            queryType {
                fields {
                    name
                    description
                    type {
                        name
                        kind
                    }
                }
            }
        }
    }
    """
    
    payload = {"query": introspection_query}
    
    try:
        res = requests.post(url, headers=headers, json=payload)
        res.raise_for_status()
        data = res.json()
        
        if "data" in data and data["data"]["__schema"]:
            query_fields = data["data"]["__schema"]["queryType"]["fields"]
            
            # Look for equipment, tool, asset related queries
            keywords = ['equipment', 'tool', 'asset', 'machine', 'gage', 'gauge']
            related_queries = []
            
            logging.info("\n" + "=" * 100)
            logging.info("EQUIPMENT-RELATED QUERIES FOUND:")
            logging.info("=" * 100)
            
            for query in query_fields:
                query_name = query['name'].lower()
                if any(keyword in query_name for keyword in keywords):
                    related_queries.append(query['name'])
                    logging.info(f"  🔧 {query['name']:<40} → {query['type'].get('name', 'Unknown')}")
            
            if not related_queries:
                logging.warning("⚠️  No equipment-related queries found")
                logging.info("\nAll available queries:")
                for query in sorted(query_fields, key=lambda x: x['name']):
                    logging.info(f"  - {query['name']}")
            
            # Save all queries
            with open('all_queries.json', 'w') as f:
                json.dump([q['name'] for q in query_fields], f, indent=2)
            
            logging.info("\n📁 All queries saved to: all_queries.json")
            
    except Exception as e:
        logging.error(f"Error discovering queries: {e}")


def fetch_equipment_data(token, field_names, query_name="equipment"):
    """Fetch equipment data using discovered fields"""
    url = f"https://msp.adionsystems.com/api/graphql?token={token}"
    headers = {"content-type": "application/json"}
    
    # Build query with all discovered fields
    fields_str = "\n                ".join(field_names)
    
    query = f"""
    query {query_name}($pageSize: Int, $pageStart: Int) {{
        {query_name}(pageSize: $pageSize, pageStart: $pageStart) {{
            records {{
                {fields_str}
            }}
            totalRecords
        }}
    }}
    """
    
    logging.info("\n" + "=" * 100)
    logging.info("FETCHING EQUIPMENT DATA...")
    logging.info("=" * 100)
    
    page_start = 0
    page_size = 1000
    all_records = []
    
    while True:
        payload = {
            "query": query,
            "variables": {
                "pageSize": page_size,
                "pageStart": page_start
            }
        }
        
        try:
            res = requests.post(url, headers=headers, json=payload)
            res.raise_for_status()
            
            response_data = res.json()
            
            if "errors" in response_data:
                logging.error(f"GraphQL errors: {response_data['errors']}")
                break
            
            data = response_data["data"][query_name]
            records = data.get("records", [])
            total_records = data.get("totalRecords", 0)
            
            logging.info(f"Fetched {len(records)} records (pageStart={page_start})")
            
            if not records:
                break
            
            all_records.extend(records)
            page_start += page_size
            
            if page_start >= total_records:
                break
                
        except Exception as e:
            logging.error(f"Error fetching data: {e}")
            break
    
    logging.info(f"\n✅ Total equipment records fetched: {len(all_records)}")
    
    if all_records:
        # Display sample record
        logging.info("\nSample Equipment Record:")
        logging.info(json.dumps(all_records[0], indent=2))
        
        # Convert to DataFrame
        df = pd.DataFrame(all_records)
        
        # Show summary
        logging.info(f"\n{'='*100}")
        logging.info("DATA SUMMARY")
        logging.info("=" * 100)
        logging.info(f"\nTotal Records: {len(df)}")
        logging.info(f"Total Columns: {len(df.columns)}")
        
        logging.info("\nColumn Summary:")
        for col in df.columns:
            non_null = df[col].notna().sum()
            logging.info(f"  {col:<40} {non_null:>6} non-null values")
        
        # Save to CSV
        csv_file = 'equipment_export.csv'
        df.to_csv(csv_file, index=False)
        logging.info(f"\n📁 Data exported to: {csv_file}")
        
        # Save sample to JSON
        with open('equipment_sample.json', 'w') as f:
            json.dump(all_records[:5], f, indent=2)
        logging.info(f"📁 Sample records saved to: equipment_sample.json")
        
        return df
    
    return None


def main():
    """Main execution flow"""
    token = get_token()
    
    if not token:
        logging.error("Failed to get authentication token")
        return
    
    # Step 1: Discover schema
    logging.info("=" * 100)
    logging.info("STEP 1: DISCOVERING EQUIPMENT SCHEMA")
    logging.info("=" * 100)
    
    field_names, type_name = introspect_equipment_schema(token)
    
    if not field_names:
        logging.error("\n❌ Could not discover equipment fields")
        return
    
    # Step 2: Fetch data
    logging.info("\n" + "=" * 100)
    logging.info("STEP 2: FETCHING EQUIPMENT DATA")
    logging.info("=" * 100)
    
    # Try different query names
    query_names = ["equipment", "tools", "internalTools", "assets"]
    
    df = None
    for query_name in query_names:
        logging.info(f"\nTrying query: {query_name}")
        try:
            df = fetch_equipment_data(token, field_names, query_name)
            if df is not None:
                logging.info(f"\n✅ SUCCESS using query: {query_name}")
                break
        except Exception as e:
            logging.debug(f"Query '{query_name}' failed: {e}")
            continue
    
    if df is None:
        logging.error("\n❌ Could not fetch equipment data with any query name")
    else:
        logging.info("\n" + "=" * 100)
        logging.info("🎉 EQUIPMENT DATA SUCCESSFULLY RETRIEVED")
        logging.info("=" * 100)
        
        # Display preview matching the UI
        logging.info("\nEquipment Preview (matching UI columns):")
        preview_cols = []
        
        # Map UI columns to possible field names
        column_mapping = {
            'Internal Tool #': ['internalToolNumber', 'toolNumber', 'equipmentNumber', 'id'],
            'Status': ['status'],
            'Brand': ['brand', 'manufacturer'],
            'Equipment Prefix': ['equipmentPrefix', 'prefix'],
            'Location': ['location'],
            'Type': ['type', 'equipmentType'],
            'Check Due': ['checkDue', 'nextCheckDate', 'calibrationDueDate', 'nextMaintenanceDate']
        }
        
        for ui_col, possible_fields in column_mapping.items():
            for field in possible_fields:
                if field in df.columns:
                    preview_cols.append(field)
                    break
        
        if preview_cols:
            print("\n" + df[preview_cols].head(20).to_string())


if __name__ == "__main__":
    main()

ERROR:root:❌ Could not find Equipment schema with any type name
ERROR:root:
❌ Could not discover equipment fields
